# 07 — Comparação: Dados do Cliente vs. Combinado

**Pergunta central:** *Os dados do cliente sozinhos já são suficientes para um bom modelo?  
Ou adicionar dados públicos melhora significativamente?*

Pré-requisitos: executar **05b_train_client_only** e **06** após o pipeline em `README.md` (idealmente com **05a_train_public_only** antes do **02** para melhor pré-rotulagem).

Este notebook carrega os resultados dos dois experimentos e responde com:
- Tabela comparativa de métricas
- Gráficos de mAP, Precision, Recall por experimento
- Comparação por classe (violence / pre_violence)
- Teste visual lado a lado nas mesmas imagens do cliente
- **Recomendação final** com base nos números

In [ ]:
!pip install -q ultralytics opencv-python-headless matplotlib seaborn pandas

In [ ]:
import json, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML
from ultralytics import YOLO

MODELS_DIR   = Path("../models")
CLIENT_YAML  = Path("../data/datasets/client_only/dataset.yaml")
CLASS_NAMES  = ["person", "violence", "pre_violence"]

r_client = json.load(open(MODELS_DIR/"results_client_only.json"))
r_comb   = json.load(open(MODELS_DIR/"results_combined.json"))
print("✓ Resultados carregados")

## 1. Tabela comparativa

In [ ]:
# Métricas gerais — avaliadas no teste do CLIENTE (mesmo conjunto para ambos)
def safe(d, *keys, default=None):
    v = d
    for k in keys:
        if not isinstance(v, dict): return default
        v = v.get(k, default)
    return v

rows = [
    {"Experimento":  "Client Only",
     "mAP@50":       r_client.get("map50"),
     "mAP@50-95":    r_client.get("map50_95"),
     "Precision":    r_client.get("precision"),
     "Recall":       r_client.get("recall"),
     "AP50 violence":     safe(r_client,"ap50_per_class","violence"),
     "AP50 pre_violence": safe(r_client,"ap50_per_class","pre_violence")},
    {"Experimento":  "Combined",
     "mAP@50":       r_comb.get("client_map50"),      # avaliado no teste do cliente
     "mAP@50-95":    r_comb.get("client_map50_95"),
     "Precision":    r_comb.get("client_precision"),
     "Recall":       r_comb.get("client_recall"),
     "AP50 violence":     safe(r_comb,"client_ap50_per_class","violence"),
     "AP50 pre_violence": safe(r_comb,"client_ap50_per_class","pre_violence")},
]

df_cmp = pd.DataFrame(rows).set_index("Experimento")
df_cmp = df_cmp.map(lambda x: f"{x:.4f}" if isinstance(x,(float,int)) and x is not None else "—")

# Highlight melhor em verde
display(HTML("<h3>Métricas no conjunto de teste do CLIENTE</h3>"))
display(df_cmp.style.set_table_styles([
    {"selector":"th","props":[("background","#2c3e50"),("color","white"),("font-size","13px")]},
    {"selector":"td","props":[("font-size","13px"),("text-align","center")]},
]))

## 2. Gráfico comparativo de métricas

In [ ]:
metrics = ["mAP@50","mAP@50-95","Precision","Recall",
           "AP50 violence","AP50 pre_violence"]

def parse(s):
    try: return float(s)
    except: return 0.0

vals_c = [parse(df_cmp.loc["Client Only", m]) for m in metrics]
vals_b = [parse(df_cmp.loc["Combined",    m]) for m in metrics]

x    = np.arange(len(metrics))
w    = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
bars_c = ax.bar(x - w/2, vals_c, w, label="Client Only", color="steelblue",   edgecolor="black", alpha=0.85)
bars_b = ax.bar(x + w/2, vals_b, w, label="Combined",    color="darkorange",  edgecolor="black", alpha=0.85)

for bar in list(bars_c) + list(bars_b):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Comparação: Client Only vs. Combined\n(avaliado no conjunto de teste do CLIENTE)",
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../models/comparison_chart.png", bbox_inches="tight")
plt.show()

## 3. Teste visual lado a lado — mesmos vídeos do cliente

In [ ]:
w_client = MODELS_DIR / "client_only_best.pt"
w_comb   = MODELS_DIR / "combined_best.pt"

if not w_client.exists() or not w_comb.exists():
    print("⚠ Pesos não encontrados. Execute os notebooks 05 e 06 primeiro.")
else:
    m_c = YOLO(str(w_client))
    m_b = YOLO(str(w_comb))

    test_imgs = list((Path("../data/datasets/client_only/images/test")).glob("*.jpg"))[:6]
    COLORS    = {0:(0,200,0), 1:(0,0,255), 2:(0,140,255)}

    def draw(frame, preds):
        vis = frame.copy()
        for box in preds.boxes:
            cls, conf = int(box.cls[0]), float(box.conf[0])
            x1,y1,x2,y2 = (int(v) for v in box.xyxy[0].tolist())
            col = COLORS.get(cls,(128,128,128))
            cv2.rectangle(vis,(x1,y1),(x2,y2),col,2)
            cv2.putText(vis,f"{CLASS_NAMES[cls]} {conf:.2f}",(x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,col,1)
        return vis

    fig, axes = plt.subplots(len(test_imgs), 2, figsize=(12, len(test_imgs)*3.5))
    if len(test_imgs)==1: axes=[axes]
    axes[0][0].set_title("Client Only", fontsize=12, fontweight="bold")
    axes[0][1].set_title("Combined",    fontsize=12, fontweight="bold")

    for row_axes, img_path in zip(axes, test_imgs):
        frame = cv2.imread(str(img_path))
        p_c   = m_c(frame, verbose=False, conf=0.3)[0]
        p_b   = m_b(frame, verbose=False, conf=0.3)[0]

        for ax, pred, model_name in [(row_axes[0],p_c,"Client"),(row_axes[1],p_b,"Combined")]:
            vis = draw(frame, pred)
            ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            ax.axis("off")
            n_v  = sum(1 for b in pred.boxes if int(b.cls[0])==1)
            n_pv = sum(1 for b in pred.boxes if int(b.cls[0])==2)
            ax.set_xlabel(f"violence={n_v} pre_v={n_pv}", fontsize=8)

    plt.suptitle("Predições: Client Only vs. Combined — Imagens do Cliente",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../models/comparison_visual.png", bbox_inches="tight", dpi=80)
    plt.show()

## 4. Recomendação final

In [ ]:
map_c = r_client.get("map50", 0) or 0
map_b = r_comb.get("client_map50", 0) or 0
diff  = map_b - map_c

if diff >= 0.05:
    rec = "COMBINED"
    msg = f"O modelo combinado é {diff*100:.1f}pp melhor. Recomendado usar dados públicos + cliente."
elif diff >= 0.01:
    rec = "COMBINED (marginal)"
    msg = f"Melhora de {diff*100:.1f}pp com dados combinados — ganho pequeno, avalie o custo de uso de dados públicos."
else:
    rec = "CLIENT ONLY"
    msg = f"Diferença de apenas {diff*100:.1f}pp. Os dados do cliente já são suficientes — não há necessidade dos dados públicos."

color = "#e74c3c" if rec.startswith("COMBINED") else "#27ae60"
display(HTML(f"""
<div style='background:{color};color:white;padding:16px;border-radius:8px;
     font-size:15px;line-height:2'>
<b>RECOMENDAÇÃO: {rec}</b><br>
{msg}<br><br>
mAP@50 Client Only : {map_c:.4f}<br>
mAP@50 Combined    : {map_b:.4f}<br>
Diferença          : {diff:+.4f}
</div>
"""))

print("\nPróximos passos:")
print("  08_lstm_pre_violence.ipynb  — treinar LSTM temporal")
print("  09_inference_demo.ipynb     — testar inferência nos vídeos do cliente")